## Data Collection and Preprocessing

Daily financial time series data are collected from Yahoo Finance using the `yfinance` library. The dataset includes Bitcoin prices together with key macro-financial variables such as gold, oil, the S&P 500 index, and exchange rates.

All series are aligned on a daily calendar, missing values are forward-filled, and the EUR/USD exchange rate is inverted to obtain USD/EUR. This results in a consistent and complete dataset suitable for subsequent analysis.

In [10]:
import os, random
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import openpyxl

import warnings
warnings.filterwarnings("ignore")


In [2]:

START = "2016-04-29"
END   = "2025-06-30"

full_days = pd.date_range(start=START, end=END, freq="D")

tickers = {
    "Bitcoin_Price": "BTC-USD",
    "Gold_Price": "GC=F",
    "Oil_Price": "CL=F",
    "SP500_Price": "^GSPC",
    "USD_CNY_Price": "CNY=X",
    "EURUSD_raw": "EURUSD=X",
}

In [3]:
raw = {}

for col_name, y_ticker in tickers.items():
    df_download = yf.download(
        y_ticker,
        start=START,
        end=END,
        progress=False,
        auto_adjust=False,
        actions=False
    )

    if df_download.empty:
        s = pd.Series(dtype="float64")
    elif "Close" in df_download:
        s = df_download["Close"].copy()
    elif "Adj Close" in df_download:
        s = df_download["Adj Close"].copy()
    else:
        s = df_download.iloc[:, -1].copy()

    s.name = y_ticker
    raw[col_name] = s

In [4]:
series = {k: v.iloc[:, 0].rename(k) for k, v in raw.items()}

df = pd.concat(series, axis=1).sort_index()
df.index = pd.to_datetime(df.index)
df.index.name = "Date"

df = df.sort_index().reindex(full_days)
df_ffill = df.ffill()

df_ffill["USD_EUR_Price"] = 1.0 / df_ffill["EURUSD_raw"]

In [5]:
data_price = df_ffill.drop(columns=["EURUSD_raw"]).rename(columns={
    "Bitcoin_Price": "Bitcoin_Price",
    "Gold_Price": "Gold_Price",
    "Oil_Price": "Oil_Price",
    "SP500_Price": "SP500_Price",
    "USD_CNY_Price": "USD_CNY_Price",
})

data_price = data_price.reset_index().rename(columns={"index": "Date"})

In [6]:
return_data = np.log(data_price.iloc[:, 1:] / data_price.iloc[:, 1:].shift(1)) * 100
return_data.index = data_price["Date"]
return_data = return_data.iloc[1:,]

In [7]:
return_data

,Bitcoin_Price,Gold_Price,Oil_Price,SP500_Price,USD_CNY_Price,USD_EUR_Price
Date,,,,,,
2016-04-30,-1.500776,0.000000,0.000000,0.000000,0.000000,0.000000
2016-05-01,0.790281,0.000000,0.000000,0.000000,0.000000,0.000000
2016-05-02,-1.607539,0.425714,-2.513913,0.777961,-0.027843,-0.938984
2016-05-03,1.259268,-0.309430,-2.555827,-0.871450,0.000000,-0.539067
2016-05-04,-0.798643,-1.357268,0.297375,-0.595458,0.273449,0.181933
...,...,...,...,...,...,...
2025-06-26,-0.374446,0.192172,0.491703,0.798813,0.071095,-0.613846
2025-06-27,0.120001,-1.810197,0.428264,0.520540,-0.124097,-0.072466
2025-06-28,0.223186,0.000000,0.000000,0.000000,0.000000,0.000000


In [8]:
data_price.index = data_price["Date"]
data_price = data_price.drop("Date", axis=1)

In [9]:
data_price

,Bitcoin_Price,Gold_Price,Oil_Price,SP500_Price,USD_CNY_Price,USD_EUR_Price
Date,,,,,,
2016-04-29,455.096985,1289.199951,45.919998,2065.300049,6.4659,0.88060
2016-04-30,448.317993,1289.199951,45.919998,2065.300049,6.4659,0.88060
2016-05-01,451.875000,1289.199951,45.919998,2065.300049,6.4659,0.88060
2016-05-02,444.669006,1294.699951,44.779999,2081.429932,6.4641,0.87237
2016-05-03,450.303986,1290.699951,43.650002,2063.370117,6.4641,0.86768
...,...,...,...,...,...,...
2025-06-26,106960.000000,3333.500000,65.239998,6141.020020,7.1764,0.85589
2025-06-27,107088.429688,3273.699951,65.519997,6173.069824,7.1675,0.85527
2025-06-28,107327.703125,3273.699951,65.519997,6173.069824,7.1675,0.85527


## Export data

The processed dataset is exported to an Excel file for external use and reproducibility. The date index is formatted as a string (YYYY-MM-DD) and stored as the first column to ensure compatibility with standard data analysis tools.

In [ ]:
df_export = return_data.copy()

df_export.index = pd.to_datetime(df_export.index)
df_export.index = df_export.index.strftime("%Y-%m-%d")
df_export.index.name = "Date"

df_export.to_excel("return_data.xlsx")

In [ ]:
df_export = data_price.copy()

df_export.index = pd.to_datetime(df_export.index)
df_export.index = df_export.index.strftime("%Y-%m-%d")
df_export.index.name = "Date"

df_export.to_excel("data_price.xlsx")